# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import logging
import torch
from hydra.utils import instantiate
from omegaconf import OmegaConf

from src.utils.notebook_setup import init_nlp_notebook


# 1. Вычисляем корень проекта ОДИН РАЗ
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# 2. Регистрируем все резолверы ГЛОБАЛЬНО
# Теперь ${paths.data_dir} будет подтягиваться из конфига,
# а если Hydra падает, мы подставляем ${PROJECT_ROOT}
OmegaConf.register_new_resolver("project_root", lambda: str(PROJECT_ROOT))
OmegaConf.register_new_resolver("hydra", lambda path: str(PROJECT_ROOT), replace=True)
OmegaConf.register_new_resolver("now", lambda *args: "now", replace=True)

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

# 3. Инициализируем Hydra
cfg = init_nlp_notebook()

# 4. Трюк: принудительно подменяем переменную paths, если она не разрешилась
# Это поможет конфигу увидеть, где лежат данные, без правок yaml-файлов
if "paths" not in cfg:
    cfg.paths = OmegaConf.create()
cfg.paths.data_dir = str(PROJECT_ROOT / "data")

device = "cuda" if torch.cuda.is_available() else "cpu"

NLP Environment ready. Root: c:\condratory_kaggle


# Data init and collator

In [2]:
from src.core.data.builder import NLPDataModule  # noqa: E402


tokenizer = instantiate(cfg.model.tokenizer).build()

datamodule = NLPDataModule(
    data_cfg=cfg.data,
    tokenizer=tokenizer
)

datamodule.prepare_data()
datamodule.setup(stage="validate")
val_dataloader = datamodule.val_dataloader()
sample_batch = next(iter(val_dataloader))

print("Keys in batch:", sample_batch.keys())
print("Input IDs shape:", sample_batch["input_ids"].shape)
if "labels" in sample_batch:
    print("Labels shape:", sample_batch["labels"].shape)

c:\condratory_kaggle\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:src.core.models.tokenization:Загрузка токенизатора: DeepPavlov/rubert-base-cased
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:h

Keys in batch: KeysView({'input_ids': tensor([[  101, 12242, 19199,   259, 11061,   259,   489,   268,   489,  9833,
           128,   256,   432, 11794,   474,  3834,   389,   261,   474,  8017,
         11947,   474,   273,   274, 89482,  4676,   489,   389,  4676,   245,
           489, 11286, 12658, 31180,   273,   532, 53964,  3733, 10835, 20159,
         10961, 16106, 11659, 10835, 16120,   273, 10651, 15698, 19855, 59753,
         10835, 11061, 11403,   489,   271, 10881, 11191, 12032,   283, 13861,
         11051, 26241,  3733,   102],
        [  101, 12463, 11743, 10701, 13308, 43677, 10859,   232, 68943,   263,
           248, 23552, 13776, 11201, 13308, 43677, 10661, 10701, 19865, 12463,
         11041, 20068, 10892, 17257, 12285, 14915, 11743, 33162, 13967, 12665,
         10701, 12463,   118, 12151, 12295,  7729, 10626, 14607, 11462, 10859,
         92261, 10617, 12084, 10037, 11043, 10616, 10937, 10765, 10623, 10783,
           118,   268, 13998, 10966, 12295,  7729, 1062

# Model init

In [ ]:
# Правильная проверка forward pass через NLPModel
base_model = instantiate(cfg.model.builder, tokenizer=tokenizer).build()
nlp_model = instantiate(cfg.model_module, model=base_model)
nlp_model.to(device)
nlp_model.eval()

INFO:src.core.models.builder:Запрос модели 'models:/watsonDetector@Production' из MLflow Model Registry...
INFO:src.core.models.builder:Загрузка модели из: DeepPavlov/rubert-base-cased
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/config.json "HTTP/1.1 200 OK"


trainable params: 297,219 || all params: 178,152,966 || trainable%: 0.1668


NLPModel(
  (model): PeftModelForSequenceClassification(
    (base_model): LoraModel(
      (model): BertForSequenceClassification(
        (bert): BertModel(
          (embeddings): BertEmbeddings(
            (word_embeddings): Embedding(119547, 768, padding_idx=0)
            (position_embeddings): Embedding(512, 768)
            (token_type_embeddings): Embedding(2, 768)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (encoder): BertEncoder(
            (layer): ModuleList(
              (0-11): 12 x BertLayer(
                (attention): BertAttention(
                  (self): BertSelfAttention(
                    (query): lora.Linear(
                      (base_layer): Linear(in_features=768, out_features=768, bias=True)
                      (lora_dropout): ModuleDict(
                        (default): Dropout(p=0.1, inplace=False)
                      )
   

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/DeepPavlov/rubert-base-cased/discussions?p=0 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/DeepPavlov/rubert-base-cased/commits/refs%2Fpr%2F4 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/refs%2Fpr%2F4/model.safetensors.index.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/refs%2Fpr%2F4/model.safetensors "HTTP/1.1 302 Found"


In [4]:
print("=== Проверка совместимости модели и данных ===\n")

# 1. Параметры модели
total_params = sum(p.numel() for p in nlp_model.parameters())
trainable_params = sum(p.numel() for p in nlp_model.parameters() if p.requires_grad)
print(f"Всего параметров:    {total_params:,}")
print(f"Обучаемых:           {trainable_params:,}")
print(f"Заморожено:          {total_params - trainable_params:,}")
print(f"% обучаемых:         {trainable_params/total_params*100:.1f}%\n")

# 2. Что именно обучается
print("Обучаемые слои:")
for name, param in nlp_model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {list(param.shape)}")

# Ожидание для LoRA: только lora_A, lora_B слои + classifier
# Если список пустой — LoRA не применилась

=== Проверка совместимости модели и данных ===

Всего параметров:    178,152,966
Обучаемых:           297,219
Заморожено:          177,855,747
% обучаемых:         0.2%

Обучаемые слои:
  model.base_model.model.bert.encoder.layer.0.attention.self.query.lora_A.default.weight: [8, 768]
  model.base_model.model.bert.encoder.layer.0.attention.self.query.lora_B.default.weight: [768, 8]
  model.base_model.model.bert.encoder.layer.0.attention.self.value.lora_A.default.weight: [8, 768]
  model.base_model.model.bert.encoder.layer.0.attention.self.value.lora_B.default.weight: [768, 8]
  model.base_model.model.bert.encoder.layer.1.attention.self.query.lora_A.default.weight: [8, 768]
  model.base_model.model.bert.encoder.layer.1.attention.self.query.lora_B.default.weight: [768, 8]
  model.base_model.model.bert.encoder.layer.1.attention.self.value.lora_A.default.weight: [8, 768]
  model.base_model.model.bert.encoder.layer.1.attention.self.value.lora_B.default.weight: [768, 8]
  model.base_model.mod

# Forward Pass

In [5]:
sample_batch_gpu = {
    k: v.to(device) for k, v in sample_batch.items()
    if isinstance(v, torch.Tensor)
}

with torch.no_grad():
    outputs = nlp_model(**sample_batch_gpu)

# BertForSequenceClassification возвращает SequenceClassifierOutput
print("Output type:", type(outputs))
print("Has logits:", hasattr(outputs, "logits"))
print("Logits shape:", outputs.logits.shape)
# Ожидание: torch.Size([16, 2]) — batch_size x num_classes

# Проверяем что предсказания дают классы 0 и 1, а не случайные числа
preds = torch.argmax(outputs.logits, dim=-1)
print("Predictions:", preds)
print("Unique predicted classes:", preds.unique().tolist())
# Ожидание: только [0] или [1] или [0, 1]

print("Labels:", sample_batch_gpu["labels"])

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Output type: <class 'transformers.modeling_outputs.SequenceClassifierOutput'>
Has logits: True
Logits shape: torch.Size([4, 3])
Predictions: tensor([2, 2, 2, 2])
Unique predicted classes: [2]
Labels: tensor([1, 2, 1, 0])


# Baseline Evaluation

In [7]:
from sklearn.metrics import classification_report  # noqa: E402
from tqdm.auto import tqdm  # noqa: E402


all_preds = []
all_labels = []
max_batches = 50

for i, batch in enumerate(tqdm(
    val_dataloader, desc="Baseline Eval",
    total=min(len(val_dataloader), max_batches)
)):
    if i >= max_batches:
        break

    batch_gpu = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        outputs = nlp_model(**batch_gpu)

    # ИСПРАВЛЕНО: берём logits, а не pooler_output
    preds = torch.argmax(outputs.logits, dim=-1)

    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(batch["labels"].cpu().numpy())

print("\n--- UNTRAINED BASELINE REPORT ---")
print(classification_report(all_labels, all_preds,
      target_names=["entailment  (0)", "neutral  (1)", "contradiction  (2)"], zero_division=0))

Baseline Eval: 100%|██████████| 50/50 [00:09<00:00,  5.17it/s]


--- UNTRAINED BASELINE REPORT ---
                    precision    recall  f1-score   support

   entailment  (0)       0.28      0.34      0.31        56
      neutral  (1)       0.25      0.04      0.07        74
contradiction  (2)       0.33      0.57      0.42        70

          accuracy                           0.31       200
         macro avg       0.29      0.32      0.27       200
      weighted avg       0.29      0.31      0.26       200



# Lora 
Все 12 слоев берта обернуты в query и value матрицы 8 ранга, классификационная голова также обучаема.(веса не заморожены)
# Classifier Head 
классификационная голова инициализирована случайно
# Baseline Metrics
Случайное угадывание с учетом дисбаланса классов, все корректно при условии необученной модели.
Отсутствует Data Leakage т.к. accuracy не завышен